In [2]:
# ================================================
# Notebook 02: Preprocesamiento NLP
# Autor: Steeven Quezada
# ================================================

import pandas as pd
import sys
import time
sys.path.append('../src')

from preprocessing import preprocess

# ================================================
# PASO 1: CARGA DE DATOS
# ================================================

# Cargar dataset limpio del EDA
df = pd.read_csv('../data/processed/dataset_final.csv')
print(f"Dataset cargado: {len(df):,} registros")
print(f"Columnas: {df.columns.tolist()}")

Dataset cargado: 38,473 registros
Columnas: ['title', 'text', 'subject', 'date', 'label', 'text_length', 'title_length', 'date_parsed', 'month', 'exclamations', 'questions', 'caps_ratio', 'avg_word_len']


In [ ]:
# ================================================
# PASO 2: APLICAR PIPELINE NLP
# ================================================
import os

# Si ya existe el dataset preprocesado, cargarlo directamente
# para evitar reprocesar 31 minutos innecesariamente
if os.path.exists('../data/processed/dataset_preprocessed.csv'):
    print("Dataset preprocesado encontrado — cargando directamente...")
    df = pd.read_csv('../data/processed/dataset_preprocessed.csv')
    print(f"✓ Cargado en segundos. Registros: {len(df):,}")
    print(f"Columnas: {df.columns.tolist()}")
else:
    print("Preprocesando textos por primera vez...")
    print("Esto tomará ~31 minutos — es normal.\n")
    
    start = time.time()
    
    df['text_clean'] = df['text'].apply(
        lambda x: preprocess(str(x), lemmatize=True)
    )
    df['title_clean'] = df['title'].apply(
        lambda x: preprocess(str(x), lemmatize=True)
    )
    
    elapsed = time.time() - start
    print(f"✓ Preprocesamiento completado en {elapsed/60:.1f} minutos")

print(f"\nEjemplo de transformación:")
print(f"\nTEXTO ORIGINAL:\n{df['text'].iloc[0][:200]}")
print(f"\nTEXTO LIMPIO:\n{df['text_clean'].iloc[0][:200]}")

Preprocesando textos por primera vez...
Esto tomará ~47 minutos — es normal.

✓ Preprocesamiento completado en 31.1 minutos

Ejemplo de transformación:

TEXTO ORIGINAL:
Donald Trump just couldn t wish all Americans a Happy New Year and leave it at that. Instead, he had to give a shout out to his enemies, haters and  the very dishonest fake news media.  The former rea

TEXTO LIMPIO:
donald trump wish americans happy new year leave instead give shout enemy hater dishonest fake news medium former reality show star one job country rapidly grow strong smart want wish friend supporter


In [5]:
# ================================================
# PASO 3: VERIFICACIÓN, COMBINACIÓN Y GUARDADO
# ================================================

# Verificar calidad del preprocesamiento
print("=== VERIFICACIÓN ===")
print(f"Textos vacíos después de preprocesamiento: "
      f"{(df['text_clean'].str.strip() == '').sum()}")

# Longitud después del preprocesamiento
df['clean_length'] = df['text_clean'].str.split().str.len()

print(f"\nLongitud promedio antes:  {df['text_length'].mean():.0f} palabras")
print(f"Longitud promedio después: {df['clean_length'].mean():.0f} palabras")
print(f"Reducción: "
      f"{(1 - df['clean_length'].mean()/df['text_length'].mean())*100:.1f}%")

# ================================================
# TRES ENFOQUES PARA MODELADO
# ================================================
print("\n=== CREANDO TRES ENFOQUES DE ENTRADA ===")

# Enfoque 1 — Solo texto
# Ya existe como text_clean

# Enfoque 2 — Solo título
# Ya existe como title_clean

# Enfoque 3 — Título + Texto combinados
df['title_text_clean'] = df['title_clean'] + ' ' + df['text_clean']

print(f"Enfoque 1 - Solo texto:           {df['text_clean'].str.split().str.len().mean():.0f} palabras promedio")
print(f"Enfoque 2 - Solo título:          {df['title_clean'].str.split().str.len().mean():.0f} palabras promedio")
print(f"Enfoque 3 - Título + Texto:       {df['title_text_clean'].str.split().str.len().mean():.0f} palabras promedio")

# Ejemplo de cada enfoque
print(f"\n=== EJEMPLO DE TRANSFORMACIÓN ===")
print(f"\nENFOQUE 1 - Solo texto:\n{df['text_clean'].iloc[0][:150]}")
print(f"\nENFOQUE 2 - Solo título:\n{df['title_clean'].iloc[0]}")
print(f"\nENFOQUE 3 - Título + Texto:\n{df['title_text_clean'].iloc[0][:150]}")

# Guardar dataset preprocesado con los tres enfoques
df.to_csv('../data/processed/dataset_preprocessed.csv', index=False)

print(f"\n=== RESUMEN FINAL ===")
print(f"Registros: {len(df):,}")
print(f"Columnas finales: {df.columns.tolist()}")
print(f"\nColumnas para modelado:")
print(f"  • text_clean       → Enfoque 1: solo texto")
print(f"  • title_clean      → Enfoque 2: solo título")
print(f"  • title_text_clean → Enfoque 3: título + texto combinados")
print(f"  • label            → variable objetivo (0=falsa, 1=real)")
print(f"\n✓ Dataset preprocesado guardado en data/processed/")
print(f"  Tiempo de preprocesamiento: 31.1 minutos")

=== VERIFICACIÓN ===
Textos vacíos después de preprocesamiento: 0

Longitud promedio antes:  405 palabras
Longitud promedio después: 229 palabras
Reducción: 43.4%

=== CREANDO TRES ENFOQUES DE ENTRADA ===
Enfoque 1 - Solo texto:           229 palabras promedio
Enfoque 2 - Solo título:          9 palabras promedio
Enfoque 3 - Título + Texto:       238 palabras promedio

=== EJEMPLO DE TRANSFORMACIÓN ===

ENFOQUE 1 - Solo texto:
donald trump wish americans happy new year leave instead give shout enemy hater dishonest fake news medium former reality show star one job country ra

ENFOQUE 2 - Solo título:
donald trump send embarrass new year ' eve message disturb

ENFOQUE 3 - Título + Texto:
donald trump send embarrass new year ' eve message disturb donald trump wish americans happy new year leave instead give shout enemy hater dishonest f

=== RESUMEN FINAL ===
Registros: 38,473
Columnas finales: ['title', 'text', 'subject', 'date', 'label', 'text_length', 'title_length', 'date_parsed', 'm